In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
sns.set_theme(style='whitegrid')
df = pd.read_csv(r'C:\Users\PRUTHVI\OneDrive\Desktop\EL Matador AI\data\ufc_clean.csv')

In [ ]:
df.head()

checking if younger fighter wins more on average or not

In [ ]:
young_red_wins = df[(df['age_dif']<0) & (df['red_win']==1)].shape[0]
young_red_total = df[(df['age_dif']<0)].shape[0]
young_blue_wins = df[(df['age_dif']>0) & (df['red_win']==0)].shape[0]
young_blue_total = df[(df['age_dif']>0)].shape[0]

In [ ]:
total_young_wins = young_blue_wins + young_red_wins
total_age_gaps_fights = young_red_total + young_blue_total
youth_win_rate = (total_young_wins/total_age_gaps_fights)*100
print(f'The younger fighter wins {youth_win_rate:.2f}% of the times when there is an age difference')

In [ ]:
heavy_age_gap = df[abs(df['age_dif'])>=6]

young_wins_heavy = heavy_age_gap[((heavy_age_gap['age_dif']<= -6) & (heavy_age_gap['red_win']==1))| ((heavy_age_gap['age_dif']>= 6) & (heavy_age_gap['red_win']==0))].shape[0]

if heavy_age_gap.shape[0] > 0:
    rate = (young_wins_heavy/heavy_age_gap.shape[0])*100
    print(f'If the age is bigger than 6 years than the chances of younger fighter winning are {rate:.2f}%')
else:
    print('There are not enough age gaps matchups')


In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(data=df,x='age_dif',hue='red_win',multiple='stack',bins=20,palette='bwr')
plt.title('Age difference Fights')
plt.xlabel('Age difference')
plt.ylabel('Number of fights')
plt.axvline(x= 0, color = 'black',linestyle = '--')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
# 'shade=True' fills the area under the wave to make the overlap obvious
sns.kdeplot(data=df, x='age_dif', hue='red_win', fill=True, palette='bwr', common_norm=False)
plt.title('Where Wins Concentrate Based on Age Difference')
plt.xlabel('Age Difference (Negative = Red is Younger | Positive = Blue is Younger)')
plt.ylabel('Density of Wins')
plt.axvline(x=0, color='black', linestyle='--')
plt.show()

we just proved that there is no thing as youth advantage in ufc as the young fighter winnings are exactly around 50% so age is not an critical factor to our project

Now we check if odds have anything to do with chances of winning or if the underdog gets equal chance at winning

In [30]:
def favourite_won(row):
    if row['R_odds'] < row['B_odds'] and row['red_win'] == 1:
        return 1
    elif row['R_odds'] > row['B_odds'] and row['red_win'] == 0:
        return 1
    elif row['R_odds'] == row['B_odds']:
        return np.nan
    else:
        return 0
    
df['favourite_correct'] = df.apply(favourite_won,axis = 1)

fav_win_rate = df['favourite_correct'].mean()*100
print(f'UFC betting favourites have a {fav_win_rate:.2f}% chances of winning')

UFC betting favourites have a 66.55% chances of winning


In [ ]:
df['odds_dif'] = df['R_odds'] - df['B_odds']

plt.figure(figsize=(8,5))
sns.kdeplot(data=df,x='odds_dif',hue='red_win',fill=True,palette='bwr',common_norm=True)
plt.title('Betting Odds')
plt.xlabel('Odds Difference')
plt.ylabel('Density of fight')
plt.axvline(x=0,color = 'black',linestyle = '--')
plt.show()

In [ ]:
red_fav_fights = df[df['R_odds'] < df['B_odds']]
red_fav_win_rate = (red_fav_fights['red_win'] == 1).mean() * 100

blue_fav_fights = df[df['B_odds'] < df['R_odds']]
blue_fav_win_rate = (blue_fav_fights['red_win'] == 0).mean() * 100

print(f"When RED is the favorite, they win {red_fav_win_rate:.2f}% of the time.")
print(f"When BLUE is the favorite, they win {blue_fav_win_rate:.2f}% of the time.")
print(f"Total fights in sample where Red was favorite: {red_fav_fights.shape[0]}")
print(f"Total fights in sample where Blue was favorite: {blue_fav_fights.shape[0]}")

now that we know if red fighter is a favourite then he has 70 % chance of winning compared to 60% chance of a blue fighter ,that is a massive edge for betters

In [ ]:
column_to_check = ['red_win', 'R_odds', 'B_odds', 'age_dif','sig_str_dif','avg_td_dif','win_streak_dif','reach_dif','avg_sub_att_dif']
corr_matrix = df[column_to_check].corr()

plt.figure(figsize=(6,6))
sns.heatmap(corr_matrix,annot=True,cmap='coolwarm',fmt='.2f',linewidths=0.5)
plt.title("Which column matters the most")
plt.show()

ok this just proved there is no single variable that would help us determine a fixed winner we need a combination of this variable to find a optimal bet for that we have to find a suitable machine learning algorithm